# Capitolo 17 – Algoritmi ricorsivi: somma, Merge Sort, ricerca binaria, sottoarray massimo (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi_cpp.ipynb)

Gli algoritmi con pseudocodice del capitolo, eseguiti sugli esempi delle dispense: la **somma di un array** (iterativa, per dimezzamento e per decremento di uno), **Merge Sort** con `Merge`, la **ricerca binaria ricorsiva** e il problema del **sottoarray massimo** (forza bruta, divide et impera, algoritmo di Kadane). Le stampe riproducono le tracce e i numeri del capitolo (con gli indici da 0).

Il capitolo completo, con pseudocodice, dimostrazioni ed esempi, è [sul sito](https://fabiofurini.github.io/algoritmi/ricorsione/17-algoritmi-ricorsivi/).

Ogni programma è scritto in un file con la cella `%%writefile` e poi compilato ed eseguito con `g++ -std=c++17 -O2`: si eseguono le celle in ordine, dall'alto in basso. Nessuna libreria esterna.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `RecursiveSum` → `RecursiveSum` (PascalCase).

## 1. Somma di un array: SumIterative, RecursiveSum, RecursiveSumLinear

- **SumIterative**$(a,n)$: un ciclo che accumula le somme parziali. $S(n)=3n+4$, memoria extra $\Theta(1)$.
- **RecursiveSum**$(a,l,r)$: divide et impera sulle due metà (caso base $l=r$). $2n-1$ chiamate, $S(n)=7n-5$, profondità $\lceil\log_2 n\rceil+1$.
- **RecursiveSumLinear**$(a,n)$: decremento di uno: $S(n)=3n-1$, profondità $n$.

*Input*: un array di $n\ge1$ interi (per `RecursiveSum`, due indici $l\le r$). *Output*: la somma degli elementi. I contatori `steps` (righe di pseudocodice eseguite), `calls` e `maxDepth` (chiamate contemporaneamente attive) misurano le grandezze analizzate nel capitolo; in `RecursiveSum` e `RecursiveSumLinear` la `return` è unica per poter aggiornare la profondità.

Sull'array dell'esempio $a=[9,8,12,3,5,14,6]$ ($n=7$) la somma è $57$; `RecursiveSum` fa $13$ chiamate e $44$ passi, con profondità $4$. Poi si esegue la traccia della tabella del capitolo su $a=[9,8,12,3,5]$: la colonna *stack* riporta le chiamate attive.

In [1]:
%%writefile sums.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long steps = 0;   // passi eseguiti (righe di pseudocodice)
int calls = 0;         // chiamate eseguite
int depth = 0;         // chiamate attive in questo momento
int maxDepth = 0;      // profondità massima della ricorsione

void Reset()
{
    steps = 0;
    calls = 0;
    depth = 0;
    maxDepth = 0;
}

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Somma iterativa di a[0..n-1]
int SumIterative(int n, int *a)
{
    int s = 0;
    int i = 0;
    steps += 2;                 // righe 1-2: s := 0, i := 1
    while (i < n)
    {
        steps += 3;             // test vero + due righe del corpo
        s = s + a[i];
        i = i + 1;
    }
    steps += 2;                 // test falso + return
    return s;
}

// Somma di a[l..r] (indici da 0), l <= r, per divide et impera
int RecursiveSum(int *a, int l, int r)
{
    calls++;
    depth++;
    if (depth > maxDepth) maxDepth = depth;
    steps++;                    // riga 1: if l = r
    int result;
    if (l == r)
    {
        steps++;                // riga 2: return a[l] (caso base)
        result = a[l];
    }
    else
    {
        steps += 4;             // righe 3-6: m, due chiamate, return
        int m = l + (r - l) / 2;                // divide
        int s1 = RecursiveSum(a, l, m);         // conquer (metà sinistra)
        int s2 = RecursiveSum(a, m + 1, r);     // conquer (metà destra)
        result = s1 + s2;                       // combine
    }
    depth--;
    return result;
}

// Somma di a[0..n-1] per decremento di uno, n >= 1
int RecursiveSumLinear(int *a, int n)
{
    calls++;
    depth++;
    if (depth > maxDepth) maxDepth = depth;
    steps++;                    // riga 1: if n = 1
    int result;
    if (n == 1)
    {
        steps++;                // riga 2: return a[1]
        result = a[0];
    }
    else
    {
        steps += 2;             // righe 3-4: chiamata e return
        int s = RecursiveSumLinear(a, n - 1);
        result = s + a[n - 1];
    }
    depth--;
    return result;
}

// ---- traccia di RecursiveSum ----
int row = 0;
int maxStack = 0;

string Join(const vector<string> &v)
{
    string s;
    for (size_t i = 0; i < v.size(); i++)
    {
        if (i > 0) s += ",";
        s += v[i];
    }
    return s;
}

void Emit(const vector<string> &stack, const string &action)
{
    row++;
    if ((int)stack.size() > maxStack) maxStack = stack.size();
    printf("%2d | %s | %s\n", row, Join(stack).c_str(), action.c_str());
}

string P(int l, int r) { return "(" + to_string(l) + "," + to_string(r) + ")"; }

int RecursiveSumTrace(int *a, int l, int r, vector<string> &stack)
{
    stack.push_back(P(l, r));
    if (l == r)
    {
        Emit(stack, "l=r => return a[" + to_string(l) + "]=" + to_string(a[l]));
        stack.pop_back();
        return a[l];
    }
    int m = l + (r - l) / 2;
    Emit(stack, "l!=r, m=" + to_string(l) + "+(" + to_string(r) + "-" + to_string(l) + ")/2=" + to_string(m) + "; call " + P(l, m));
    int s1 = RecursiveSumTrace(a, l, m, stack);
    Emit(stack, "s1=" + to_string(s1) + "; call " + P(m + 1, r));
    int s2 = RecursiveSumTrace(a, m + 1, r, stack);
    Emit(stack, "s2=" + to_string(s2) + "; return " + to_string(s1) + "+" + to_string(s2) + "=" + to_string(s1 + s2));
    stack.pop_back();
    return s1 + s2;
}

#ifndef NOMAIN
int main()
{
    int a[] = {9, 8, 12, 3, 5, 14, 6};
    int n = 7;
    printf("a = %s   n = %d\n", Fmt(a, n).c_str(), n);
    Reset();
    int s = SumIterative(n, a);
    printf("SumIterative       = %d   steps=%lld (3n+4=%d)\n", s, steps, 3 * n + 4);
    Reset();
    s = RecursiveSum(a, 0, n - 1);
    printf("RecursiveSum       = %d   calls=%d (2n-1=%d)  steps=%lld (7n-5=%d)  maxDepth=%d\n", s, calls, 2 * n - 1, steps, 7 * n - 5, maxDepth);
    Reset();
    s = RecursiveSumLinear(a, n);
    printf("RecursiveSumLinear = %d   calls=%d  steps=%lld (3n-1=%d)  maxDepth=%d\n", s, calls, steps, 3 * n - 1, maxDepth);

    int b[] = {9, 8, 12, 3, 5};
    printf("\n");
    printf("a = %s   RecursiveSum(a, 0, 4)\n", Fmt(b, 5).c_str());
    printf(" # | stack | azione\n");
    vector<string> stack;
    int res = RecursiveSumTrace(b, 0, 4, stack);
    printf("risultato %d, righe %d, massimo numero di chiamate attive %d\n", res, row, maxStack);
    return 0;
}
#endif

Writing sums.cpp


In [2]:
!g++ -std=c++17 -O2 -o sums sums.cpp && ./sums

a = [9, 8, 12, 3, 5, 14, 6]   n = 7
SumIterative       = 57   steps=25 (3n+4=25)
RecursiveSum       = 57   calls=13 (2n-1=13)  steps=44 (7n-5=44)  maxDepth=4
RecursiveSumLinear = 57   calls=7  steps=20 (3n-1=20)  maxDepth=7

a = [9, 8, 12, 3, 5]   RecursiveSum(a, 0, 4)
 # | stack | azione
 1 | (0,4) | l!=r, m=0+(4-0)/2=2; call (0,2)
 2 | (0,4),(0,2) | l!=r, m=0+(2-0)/2=1; call (0,1)
 3 | (0,4),(0,2),(0,1) | l!=r, m=0+(1-0)/2=0; call (0,0)
 4 | (0,4),(0,2),(0,1),(0,0) | l=r => return a[0]=9
 5 | (0,4),(0,2),(0,1) | s1=9; call (1,1)
 6 | (0,4),(0,2),(0,1),(1,1) | l=r => return a[1]=8
 7 | (0,4),(0,2),(0,1) | s2=8; return 9+8=17
 8 | (0,4),(0,2) | s1=17; call (2,2)
 9 | (0,4),(0,2),(2,2) | l=r => return a[2]=12
10 | (0,4),(0,2) | s2=12; return 17+12=29
11 | (0,4) | s1=29; call (3,4)
12 | (0,4),(3,4) | l!=r, m=3+(4-3)/2=3; call (3,3)
13 | (0,4),(3,4),(3,3) | l=r => return a[3]=3
14 | (0,4),(3,4) | s1=3; call (4,4)
15 | (0,4),(3,4),(4,4) | l=r => return a[4]=5
16 | (0,4),(3,4) | s2=5; retur

**Prova tu.** Cambia l'array `a` (anche la lunghezza) e confronta passi, chiamate e profondità: solo `RecursiveSumLinear` ha profondità $n$.

In [3]:
%%writefile prova_sums.cpp
#define NOMAIN
#include "sums.cpp"

int main()
{
    // Cambia l'array: confronta passi, chiamate e profondità dei tre algoritmi.
    int a[] = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16};
    int n = 16;
    Reset(); int s = SumIterative(n, a);
    printf("%d %lld\n", s, steps);
    Reset(); s = RecursiveSum(a, 0, n - 1);
    printf("%d %lld %d %d\n", s, steps, calls, maxDepth);
    Reset(); s = RecursiveSumLinear(a, n);
    printf("%d %lld %d %d\n", s, steps, calls, maxDepth);
    return 0;
}

Writing prova_sums.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_sums prova_sums.cpp && ./prova_sums

136 52
136 107 31 5
136 47 16 16


## 2. Merge Sort e Merge

- **MergeSort**$(a,l,r)$ — *Input*: un array e due indici $l\le r$. *Output*: il sotto-array $a[l..r]$ ordinato in modo non decrescente. Si chiama `MergeSort(a, 1, n)` nello pseudocodice, `merge_sort(a, 0, n-1)` nel codice.
- **Merge**$(a,l,m,r)$ — *Input*: un array e indici $l\le m<r$ tali che $a[l..m]$ e $a[m+1..r]$ siano ordinati. *Output*: $a[l..r]$ ordinato. Copia le due metà in $L$ e $R$ e riversa in $a$, passo dopo passo, il minore fra i primi due elementi non ancora copiati.

Il contatore `comparisons` conta i confronti fra elementi eseguiti da `Merge`. Sull'array dell'esempio $a=[9,12,3,16,19,11,12]$ si stampano le chiamate di `Merge` nell'ordine di esecuzione (con l'array dopo ciascuna, tabella del capitolo; gli indici sono da 0: aggiungendo 1 si ritrovano quelli del capitolo), poi la traccia di `Merge(a, 0, 1, 3)` su $a=[5,8,2,7]$ (esempio dell'implementazione in C++).

In [5]:
%%writefile merge_sort.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long comparisons = 0;   // confronti fra elementi eseguiti da Merge
int msCalls = 0;             // chiamate di MergeSort

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Merge(a, l, m, r): fonde a[l..m] e a[m+1..r], entrambi ordinati
void Merge(int *a, int l, int m, int r)
{
    int n1 = m - l + 1;             // dimensione della parte sinistra
    int n2 = r - m;                 // dimensione della parte destra
    vector<int> L(n1), R(n2);       // array ausiliari
    for (int i = 0; i < n1; i++)
    {
        L[i] = a[l + i];
    }
    for (int j = 0; j < n2; j++)
    {
        R[j] = a[m + 1 + j];
    }
    int i = 0, j = 0, k = l;
    while (i < n1 && j < n2)        // fonde L e R in a[l..r]
    {
        comparisons++;
        if (L[i] <= R[j])
        {
            a[k] = L[i];
            i++;
        }
        else
        {
            a[k] = R[j];
            j++;
        }
        k++;
    }
    while (i < n1)                  // copia il resto di L
    {
        a[k] = L[i];
        i++;
        k++;
    }
    while (j < n2)                  // copia il resto di R
    {
        a[k] = R[j];
        j++;
        k++;
    }
}

// MergeSort(a, l, r): ordina a[l..r]
void MergeSort(int *a, int l, int r)
{
    msCalls++;
    if (l < r)
    {
        int m = l + (r - l) / 2;    // divide
        MergeSort(a, l, m);         // conquer (metà sinistra)
        MergeSort(a, m + 1, r);     // conquer (metà destra)
        Merge(a, l, m, r);          // combine
    }
}

// ---- varianti che stampano la traccia ----
void MergeSortLog(int *a, int l, int r)
{
    if (l < r)
    {
        int m = l + (r - l) / 2;
        MergeSortLog(a, l, m);
        MergeSortLog(a, m + 1, r);
        string left = Fmt(a + l, m - l + 1), right = Fmt(a + m + 1, r - m);
        Merge(a, l, m, r);
        printf("Merge(a,%d,%d,%d)  %s e %s  ->  %s\n", l, m, r, left.c_str(), right.c_str(), Fmt(a, 7).c_str());
    }
}

void MergeTrace(int *a, int n, int l, int m, int r)
{
    int n1 = m - l + 1, n2 = r - m;
    vector<int> L(a + l, a + m + 1), R(a + m + 1, a + r + 1);
    printf("Merge(a, %d, %d, %d) su a = %s\n", l, m, r, Fmt(a, n).c_str());
    printf("n1=%d, n2=%d, L=%s, R=%s\n", n1, n2, Fmt(L.data(), n1).c_str(), Fmt(R.data(), n2).c_str());
    int i = 0, j = 0, k = l, it = 0;
    while (i < n1 && j < n2)
    {
        it++;
        char head[200];
        if (L[i] <= R[j])
        {
            snprintf(head, sizeof head, "%2d. ciclo 1: L[%d]=%d <= R[%d]=%d? Sì => a[%d] = L[%d] = %d", it, i, L[i], j, R[j], k, i, L[i]);
            a[k] = L[i];
            i++;
        }
        else
        {
            snprintf(head, sizeof head, "%2d. ciclo 1: L[%d]=%d <= R[%d]=%d? No => a[%d] = R[%d] = %d", it, i, L[i], j, R[j], k, j, R[j]);
            a[k] = R[j];
            j++;
        }
        k++;
        printf("%s, i=%d, j=%d, k=%d; a=%s\n", head, i, j, k, Fmt(a, n).c_str());
    }
    while (i < n1)
    {
        a[k] = L[i];
        printf("    ciclo 2 (resto di L): a[%d] = L[%d] = %d; a=%s\n", k, i, L[i], Fmt(a, n).c_str());
        i++;
        k++;
    }
    while (j < n2)
    {
        a[k] = R[j];
        printf("    ciclo 3 (resto di R): a[%d] = R[%d] = %d; a=%s\n", k, j, R[j], Fmt(a, n).c_str());
        j++;
        k++;
    }
    printf("risultato a = %s, confronti = %d\n", Fmt(a, n).c_str(), it);
}

#ifndef NOMAIN
int main()
{
    int a[] = {9, 12, 3, 16, 19, 11, 12};
    int n = 7;
    printf("a = %s   n = %d\n", Fmt(a, n).c_str(), n);
    printf("Chiamate di Merge in ordine di esecuzione:\n");
    int copia[] = {9, 12, 3, 16, 19, 11, 12};
    MergeSortLog(copia, 0, n - 1);
    comparisons = 0;
    msCalls = 0;
    MergeSort(a, 0, n - 1);
    printf("ordinato = %s   chiamate di MergeSort=%d (2n-1=%d)   confronti=%lld\n", Fmt(a, n).c_str(), msCalls, 2 * n - 1, comparisons);
    printf("\n");
    int b[] = {5, 8, 2, 7};
    MergeTrace(b, 4, 0, 1, 3);
    return 0;
}
#endif

Writing merge_sort.cpp


In [6]:
!g++ -std=c++17 -O2 -o merge_sort merge_sort.cpp && ./merge_sort

a = [9, 12, 3, 16, 19, 11, 12]   n = 7
Chiamate di Merge in ordine di esecuzione:
Merge(a,0,0,1)  [9] e [12]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,2,2,3)  [3] e [16]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,0,1,3)  [9, 12] e [3, 16]  ->  [3, 9, 12, 16, 19, 11, 12]
Merge(a,4,4,5)  [19] e [11]  ->  [3, 9, 12, 16, 11, 19, 12]
Merge(a,4,5,6)  [11, 19] e [12]  ->  [3, 9, 12, 16, 11, 12, 19]
Merge(a,0,3,6)  [3, 9, 12, 16] e [11, 12, 19]  ->  [3, 9, 11, 12, 12, 16, 19]
ordinato = [3, 9, 11, 12, 12, 16, 19]   chiamate di MergeSort=13 (2n-1=13)   confronti=14

Merge(a, 0, 1, 3) su a = [5, 8, 2, 7]
n1=2, n2=2, L=[5, 8], R=[2, 7]
 1. ciclo 1: L[0]=5 <= R[0]=2? No => a[0] = R[0] = 2, i=0, j=1, k=1; a=[2, 8, 2, 7]
 2. ciclo 1: L[0]=5 <= R[1]=7? Sì => a[1] = L[0] = 5, i=1, j=1, k=2; a=[2, 5, 2, 7]
 3. ciclo 1: L[1]=8 <= R[1]=7? No => a[2] = R[1] = 7, i=1, j=2, k=3; a=[2, 5, 7, 7]
    ciclo 2 (resto di L): a[3] = L[1] = 8; a=[2, 5, 7, 8]
risultato a = [2, 5, 7, 8], confronti = 3


**Prova tu.** Ordina un altro array: il numero di confronti di `Merge` è al massimo $n\lceil\log_2 n\rceil$ e le chiamate di `MergeSort` sono sempre $2n-1$.

In [7]:
%%writefile prova_merge_sort.cpp
#define NOMAIN
#include "merge_sort.cpp"

int main()
{
    // Ordina un altro array e conta i confronti (al massimo n*ceil(log2 n)).
    int b[] = {38, 27, 43, 3, 9, 82, 10, 5};
    int n = 8;
    comparisons = 0;
    msCalls = 0;
    MergeSort(b, 0, n - 1);
    printf("%s %lld %d\n", Fmt(b, n).c_str(), comparisons, msCalls);
    return 0;
}

Writing prova_merge_sort.cpp


In [8]:
!g++ -std=c++17 -O2 -o prova_merge_sort prova_merge_sort.cpp && ./prova_merge_sort

[3, 5, 9, 10, 27, 38, 43, 82] 17 15


## 3. RecursiveBinarySearch

**RecursiveBinarySearch**$(a,left,right,m)$ — *Input*: un array $a$ di naturali ordinato in modo non decrescente, due indici $left$ e $right$ e un valore $m$. *Output*: un indice $i\in\{left,\dots,right\}$ con $a[i]=m$, oppure $-1$ se $m$ non è in $a[left..right]$. Ogni chiamata fa **al più una** chiamata ricorsiva sulla metà (circa) dell'intervallo: l'albero delle chiamate è un cammino, con al più $\lfloor\log_2 n\rfloor+2$ chiamate. Per cercare in tutto l'array si chiama `RecursiveBinarySearch(a, 0, n-1, m)`.

Sullo stesso array $a=[3,9,11,12,12,16,19]$ del capitolo si cercano $m=12,10,3,19$. Per ogni ricerca la tabella ha una riga per chiamata ($left$, $right$, $mid$, $a[mid]$, azione), come nel capitolo ma con gli indici da 0: aggiungendo 1 si ritrovano quelli del capitolo (per esempio $m=19$ è trovato all'indice 6, cioè alla posizione 7).

In [9]:
%%writefile binary_search.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

int rbsCalls = 0;   // chiamate eseguite

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Indice di m in a[left..right] (array ordinato), oppure -1
int RecursiveBinarySearch(int *a, int left, int right, int m)
{
    rbsCalls++;
    if (left > right)
    {
        return -1;                  // intervallo vuoto: non trovato
    }
    int mid = left + (right - left) / 2;
    if (a[mid] == m)
    {
        return mid;                 // trovato
    }
    if (a[mid] < m)
    {
        return RecursiveBinarySearch(a, mid + 1, right, m);    // metà destra
    }
    return RecursiveBinarySearch(a, left, mid - 1, m);         // metà sinistra
}

// Come sopra, ma stampa una riga per chiamata (k = numero della chiamata)
int RbsTrace(int *a, int left, int right, int m, int k)
{
    if (left > right)
    {
        printf("%4d %4d %5d %3s %6s left > right => non trovato, return -1\n", k, left, right, "-", "-");
        return -1;
    }
    int mid = left + (right - left) / 2;
    printf("%4d %4d %5d %3d %6d ", k, left, right, mid, a[mid]);
    if (a[mid] == m)
    {
        printf("a[mid] = m => trovato, return %d\n", mid);
        return mid;
    }
    if (a[mid] < m)
    {
        printf("a[mid] < m => chiamata su left=%d, right=%d\n", mid + 1, right);
        return RbsTrace(a, mid + 1, right, m, k + 1);
    }
    printf("a[mid] > m => chiamata su left=%d, right=%d\n", left, mid - 1);
    return RbsTrace(a, left, mid - 1, m, k + 1);
}

#ifndef NOMAIN
int main()
{
    int a[] = {3, 9, 11, 12, 12, 16, 19};
    int n = 7;
    printf("a = %s   n = %d\n", Fmt(a, n).c_str(), n);
    int ms[] = {12, 10, 3, 19};
    for (int m : ms)
    {
        printf("\n");
        printf("Ricerca di m = %d\n", m);
        printf("%4s %4s %5s %3s %6s azione\n", "call", "left", "right", "mid", "a[mid]");
        RbsTrace(a, 0, n - 1, m, 1);
    }
    printf("\n");
    for (int m : ms)
    {
        rbsCalls = 0;
        int r = RecursiveBinarySearch(a, 0, n - 1, m);
        printf("m=%-3d -> %2d (%2d con indici da 1)  chiamate=%d\n", m, r, r >= 0 ? r + 1 : -1, rbsCalls);
    }
    return 0;
}
#endif

Writing binary_search.cpp


In [10]:
!g++ -std=c++17 -O2 -o binary_search binary_search.cpp && ./binary_search

a = [3, 9, 11, 12, 12, 16, 19]   n = 7

Ricerca di m = 12
call left right mid a[mid] azione
   1    0     6   3     12 a[mid] = m => trovato, return 3

Ricerca di m = 10
call left right mid a[mid] azione
   1    0     6   3     12 a[mid] > m => chiamata su left=0, right=2
   2    0     2   1      9 a[mid] < m => chiamata su left=2, right=2
   3    2     2   2     11 a[mid] > m => chiamata su left=2, right=1
   4    2     1   -      - left > right => non trovato, return -1

Ricerca di m = 3
call left right mid a[mid] azione
   1    0     6   3     12 a[mid] > m => chiamata su left=0, right=2
   2    0     2   1      9 a[mid] > m => chiamata su left=0, right=0
   3    0     0   0      3 a[mid] = m => trovato, return 0

Ricerca di m = 19
call left right mid a[mid] azione
   1    0     6   3     12 a[mid] < m => chiamata su left=4, right=6
   2    4     6   5     16 a[mid] < m => chiamata su left=6, right=6
   3    6     6   6     19 a[mid] = m => trovato, return 6

m=12  ->  3 ( 4 con ind

**Prova tu.** Cerca un altro valore (presente o assente) e verifica che le chiamate siano al più $\lfloor\log_2 n\rfloor+2$.

In [11]:
%%writefile prova_binary_search.cpp
#define NOMAIN
#include "binary_search.cpp"

int main()
{
    // Cerca un altro valore, anche assente: le chiamate sono al più floor(log2 n) + 2.
    int a[] = {3, 9, 11, 12, 12, 16, 19};
    int n = 7;
    rbsCalls = 0;
    int r = RecursiveBinarySearch(a, 0, n - 1, 16);
    printf("%d %d\n", r, rbsCalls);
    rbsCalls = 0;
    r = RecursiveBinarySearch(a, 0, n - 1, 100);
    printf("%d %d\n", r, rbsCalls);
    return 0;
}

Writing prova_binary_search.cpp


In [12]:
!g++ -std=c++17 -O2 -o prova_binary_search prova_binary_search.cpp && ./prova_binary_search

5 2
-1 4


## 4. BruteForceMaximumSubarray

**BruteForceMaximumSubarray**$(a,n)$ — *Input*: un array $a[1..n]$ di $n$ interi. *Output*: gli indici $bl\le br$ di un sottoarray di somma massima e la somma $best$. Esamina tutti gli $n(n+1)/2$ sottoarray $a[l..r]$ (non vuoti, di elementi consecutivi), calcolando le somme in modo incrementale: $\Theta(n^2)$.

Sull'esempio $a=[-2,5,-1,3]$ del capitolo la tabella ha una riga per sottoarray esaminato ($l$, $r$, somma, azione; in ogni riga che non migliora si indica $best$ corrente). Gli indici sono da 0: il capitolo trova $(2,4,7)$, cioè $a[2..4]=[5,-1,3]$ di somma $7$, qui $(1,3,7)$.

In [13]:
%%writefile maxsub_brute.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

struct Triple
{
    int i, j, sum;
};

// Restituisce (bl, br, best): un sottoarray di somma massima di a[0..n-1]
Triple BruteForceMaximumSubarray(int *a, int n)
{
    int best = a[0];
    int bl = 0;
    int br = 0;
    int l = 0;
    while (l < n)
    {
        int sum = 0;
        int r = l;
        while (r < n)
        {
            sum = sum + a[r];             // somma di a[l..r]
            if (sum > best)
            {
                best = sum;
                bl = l;
                br = r;
            }
            r = r + 1;
        }
        l = l + 1;
    }
    return {bl, br, best};
}

// Come sopra, ma stampa una riga per ogni sottoarray esaminato
Triple BruteForceTrace(int *a, int n)
{
    int best = a[0];
    int bl = 0;
    int br = 0;
    for (int l = 0; l < n; l++)
    {
        int sum = 0;
        for (int r = l; r < n; r++)
        {
            sum = sum + a[r];
            if (sum > best)
            {
                best = sum;
                bl = l;
                br = r;
                printf("%3d %3d %5d  sum > best => best=%d, bl=%d, br=%d\n", l, r, sum, best, bl, br);
            }
            else
            {
                printf("%3d %3d %5d  sum <= best=%d\n", l, r, sum, best);
            }
        }
    }
    return {bl, br, best};
}

#ifndef NOMAIN
int main()
{
    int a[] = {-2, 5, -1, 3};
    int n = 4;
    printf("a = %s   n = %d   sottoarray n(n+1)/2 = %d\n", Fmt(a, n).c_str(), n, n * (n + 1) / 2);
    printf("%3s %3s %5s  azione\n", "l", "r", "sum");
    Triple t = BruteForceTrace(a, n);
    printf("risultato (bl, br, best) = (%d, %d, %d)   (%d, %d, %d con indici da 1)\n", t.i, t.j, t.sum, t.i + 1, t.j + 1, t.sum);
    Triple u = BruteForceMaximumSubarray(a, n);
    printf("BruteForceMaximumSubarray(a, n) = (%d, %d, %d)\n", u.i, u.j, u.sum);
    return 0;
}
#endif

Writing maxsub_brute.cpp


In [14]:
!g++ -std=c++17 -O2 -o maxsub_brute maxsub_brute.cpp && ./maxsub_brute

a = [-2, 5, -1, 3]   n = 4   sottoarray n(n+1)/2 = 10
  l   r   sum  azione
  0   0    -2  sum <= best=-2
  0   1     3  sum > best => best=3, bl=0, br=1
  0   2     2  sum <= best=3
  0   3     5  sum > best => best=5, bl=0, br=3
  1   1     5  sum <= best=5
  1   2     4  sum <= best=5
  1   3     7  sum > best => best=7, bl=1, br=3
  2   2    -1  sum <= best=7
  2   3     2  sum <= best=7
  3   3     3  sum <= best=7
risultato (bl, br, best) = (1, 3, 7)   (2, 4, 7 con indici da 1)
BruteForceMaximumSubarray(a, n) = (1, 3, 7)


**Prova tu.** Cambia l'array (per esempio con soli numeri negativi): il sottoarray massimo è il singolo elemento maggiore.

In [15]:
%%writefile prova_maxsub_brute.cpp
#define NOMAIN
#include "maxsub_brute.cpp"

int main()
{
    // Prova con un array di soli numeri negativi: il sottoarray massimo è il singolo elemento maggiore.
    int b[] = {-3, -1, -2, -5};
    Triple t = BruteForceMaximumSubarray(b, 4);
    printf("(%d, %d, %d)\n", t.i, t.j, t.sum);
    return 0;
}

Writing prova_maxsub_brute.cpp


In [16]:
!g++ -std=c++17 -O2 -o prova_maxsub_brute prova_maxsub_brute.cpp && ./prova_maxsub_brute

(1, 1, -1)


## 5. FindMaxCrossingSubarray e FindMaximumSubarray (divide et impera)

- **FindMaxCrossingSubarray**$(a,l,mid,r)$ — *Input*: un array e tre indici $l\le mid<r$. *Output*: gli indici $maxleft\le mid<maxright$ di un sottoarray di somma massima fra quelli che attraversano il punto medio, e la sua somma. Scandisce la metà sinistra da $mid$ a $l$ e la destra da $mid+1$ a $r$: $\Theta(n)$.
- **FindMaximumSubarray**$(a,l,r)$ — *Input*: un array e due indici $l\le r$. *Output*: gli indici $i\le j$ di un sottoarray di somma massima di $a[l..r]$ e la sua somma. Il migliore fra il sottoarray massimo della metà sinistra, della destra e di quelli che attraversano il punto medio (a parità, prima il sinistro, poi il destro). $S(n)=2S(n/2)+\Theta(n)=\Theta(n\log_2 n)$, $2n-1$ chiamate.

Il $-\infty$ dello pseudocodice è il valore più piccolo rappresentabile. Prima la traccia sull'esempio $a=[-2,5,-1,3]$ (come l'esecuzione dell'implementazione C++ del capitolo; risultato $(1,3,7)$, cioè $(2,4,7)$ con indici da 1), poi l'esempio della compravendita di azioni, con le variazioni giornaliere $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$: tabella dei risultati delle chiamate sui sottointervalli (per livello dell'albero, come nel capitolo; gli indici sono da 0) e i calcoli dei due `FindMaxCrossingSubarray` del livello 1. Il risultato è $(7,10,43)$, cioè $(8,11,43)$ con indici da 1.

In [17]:
%%writefile maxsub_dc.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

int fmsCalls = 0;   // chiamate di FindMaximumSubarray

struct Triple
{
    int i, j, sum;
};

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

string Tr(const Triple &t)
{
    return "(" + to_string(t.i) + ", " + to_string(t.j) + ", " + to_string(t.sum) + ")";
}

// Restituisce (maxleft, maxright, somma) del miglior sottoarray che attraversa mid
Triple FindMaxCrossingSubarray(int *a, int l, int mid, int r)
{
    int leftsum = INT_MIN;                  // -infinito
    int sum = 0;
    int maxleft = mid;
    for (int i = mid; i >= l; i--)          // da mid a l
    {
        sum = sum + a[i];                   // somma di a[i..mid]
        if (sum > leftsum)
        {
            leftsum = sum;
            maxleft = i;
        }
    }
    int rightsum = INT_MIN;
    sum = 0;
    int maxright = mid + 1;
    for (int j = mid + 1; j <= r; j++)      // da mid+1 a r
    {
        sum = sum + a[j];                   // somma di a[mid+1..j]
        if (sum > rightsum)
        {
            rightsum = sum;
            maxright = j;
        }
    }
    return {maxleft, maxright, leftsum + rightsum};
}

// Restituisce (i, j, somma) di un sottoarray di somma massima di a[l..r]
Triple FindMaximumSubarray(int *a, int l, int r)
{
    fmsCalls++;
    if (l == r)
    {
        return {l, r, a[l]};                // caso base: un elemento
    }
    int mid = l + (r - l) / 2;
    Triple left = FindMaximumSubarray(a, l, mid);
    Triple right = FindMaximumSubarray(a, mid + 1, r);
    Triple cross = FindMaxCrossingSubarray(a, l, mid, r);
    if (left.sum >= right.sum && left.sum >= cross.sum)
    {
        return left;
    }
    if (right.sum >= left.sum && right.sum >= cross.sum)
    {
        return right;
    }
    return cross;
}

// ---- varianti che stampano la traccia ----
Triple Best(const Triple &left, const Triple &right, const Triple &cross)
{
    if (left.sum >= right.sum && left.sum >= cross.sum) return left;
    if (right.sum >= left.sum && right.sum >= cross.sum) return right;
    return cross;
}

Triple FmsTrace(int *a, int l, int r, int depth)
{
    string pad(2 * depth, ' ');
    if (l == r)
    {
        printf("%sFindMaximumSubarray(a, %d, %d) = (%d, %d, %d)  caso base\n", pad.c_str(), l, r, l, r, a[l]);
        return {l, r, a[l]};
    }
    int mid = l + (r - l) / 2;
    printf("%sFindMaximumSubarray(a, %d, %d): l != r, mid = %d\n", pad.c_str(), l, r, mid);
    Triple left = FmsTrace(a, l, mid, depth + 1);
    Triple right = FmsTrace(a, mid + 1, r, depth + 1);
    Triple cross = FindMaxCrossingSubarray(a, l, mid, r);
    printf("%s  FindMaxCrossingSubarray(a, %d, %d, %d) = %s\n", pad.c_str(), l, mid, r, Tr(cross).c_str());
    Triple ret = Best(left, right, cross);
    printf("%s  left.sum=%d, right.sum=%d, cross.sum=%d => restituisce %s\n", pad.c_str(), left.sum, right.sum, cross.sum, Tr(ret).c_str());
    return ret;
}

struct Row
{
    int l, r, mid;
    Triple left, right, cross, ret;
};

Triple FmsRows(int *a, int l, int r, vector<Row> &rows)
{
    if (l == r)
    {
        return {l, r, a[l]};
    }
    int mid = l + (r - l) / 2;
    Triple left = FmsRows(a, l, mid, rows);
    Triple right = FmsRows(a, mid + 1, r, rows);
    Triple cross = FindMaxCrossingSubarray(a, l, mid, r);
    Triple ret = Best(left, right, cross);
    rows.push_back({l, r, mid, left, right, cross, ret});
    return ret;
}

void CrossingTrace(int *a, int l, int mid, int r)
{
    printf("FindMaxCrossingSubarray(a, %d, %d, %d)\n", l, mid, r);
    printf("%3s %5s %5s %8s %8s\n", "i", "a[i]", "sum", "leftsum", "maxleft");
    int leftsum = INT_MIN, maxleft = mid, sum = 0;
    for (int i = mid; i >= l; i--)
    {
        sum = sum + a[i];
        if (sum > leftsum)
        {
            leftsum = sum;
            maxleft = i;
        }
        printf("%3d %5d %5d %8d %8d\n", i, a[i], sum, leftsum, maxleft);
    }
    printf("%3s %5s %5s %8s %8s\n", "j", "a[j]", "sum", "rightsum", "maxright");
    int rightsum = INT_MIN, maxright = mid + 1;
    sum = 0;
    for (int j = mid + 1; j <= r; j++)
    {
        sum = sum + a[j];
        if (sum > rightsum)
        {
            rightsum = sum;
            maxright = j;
        }
        printf("%3d %5d %5d %8d %8d\n", j, a[j], sum, rightsum, maxright);
    }
    printf("sottoarray che attraversa a[%d..%d], somma %d + %d = %d\n", maxleft, maxright, leftsum, rightsum, leftsum + rightsum);
}

#ifndef NOMAIN
int main()
{
    int a[] = {-2, 5, -1, 3};
    printf("a = %s\n", Fmt(a, 4).c_str());
    Triple res = FmsTrace(a, 0, 3, 0);
    fmsCalls = 0;
    FindMaximumSubarray(a, 0, 3);
    printf("risultato %s   (%d, %d, %d con indici da 1)   chiamate=%d (2n-1=%d)\n", Tr(res).c_str(), res.i + 1, res.j + 1, res.sum, fmsCalls, 2 * 4 - 1);

    printf("\n");
    int b[] = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
    int n = 16;
    printf("a = %s\n", Fmt(b, n).c_str());
    fmsCalls = 0;
    res = FindMaximumSubarray(b, 0, n - 1);
    printf("FindMaximumSubarray(a, 0, %d) = %s   (%d, %d, %d con indici da 1)   chiamate=%d (2n-1=%d)\n", n - 1, Tr(res).c_str(), res.i + 1, res.j + 1, res.sum, fmsCalls, 2 * n - 1);
    vector<Row> rows;
    FmsRows(b, 0, n - 1, rows);
    sort(rows.begin(), rows.end(), [](const Row &x, const Row &y)
         { return (x.r - x.l) != (y.r - y.l) ? (x.r - x.l) > (y.r - y.l) : x.l < y.l; });
    printf("%-9s %3s  %-13s %-13s %-13s restituito\n", "call", "mid", "left", "right", "crossing");
    for (const Row &w : rows)
    {
        string call = "[" + to_string(w.l) + ".." + to_string(w.r) + "]";
        printf("%-9s %3d  %-13s %-13s %-13s %s\n", call.c_str(), w.mid, Tr(w.left).c_str(), Tr(w.right).c_str(), Tr(w.cross).c_str(), Tr(w.ret).c_str());
    }
    printf("\n");
    CrossingTrace(b, 0, 3, 7);
    printf("\n");
    CrossingTrace(b, 8, 11, 15);
    return 0;
}
#endif

Writing maxsub_dc.cpp


In [18]:
!g++ -std=c++17 -O2 -o maxsub_dc maxsub_dc.cpp && ./maxsub_dc

a = [-2, 5, -1, 3]
FindMaximumSubarray(a, 0, 3): l != r, mid = 1
  FindMaximumSubarray(a, 0, 1): l != r, mid = 0
    FindMaximumSubarray(a, 0, 0) = (0, 0, -2)  caso base
    FindMaximumSubarray(a, 1, 1) = (1, 1, 5)  caso base
    FindMaxCrossingSubarray(a, 0, 0, 1) = (0, 1, 3)
    left.sum=-2, right.sum=5, cross.sum=3 => restituisce (1, 1, 5)
  FindMaximumSubarray(a, 2, 3): l != r, mid = 2
    FindMaximumSubarray(a, 2, 2) = (2, 2, -1)  caso base
    FindMaximumSubarray(a, 3, 3) = (3, 3, 3)  caso base
    FindMaxCrossingSubarray(a, 2, 2, 3) = (2, 3, 2)
    left.sum=-1, right.sum=3, cross.sum=2 => restituisce (3, 3, 3)
  FindMaxCrossingSubarray(a, 0, 1, 3) = (1, 3, 7)
  left.sum=5, right.sum=3, cross.sum=7 => restituisce (1, 3, 7)
risultato (1, 3, 7)   (2, 4, 7 con indici da 1)   chiamate=7 (2n-1=7)

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
FindMaximumSubarray(a, 0, 15) = (7, 10, 43)   (8, 11, 43 con indici da 1)   chiamate=31 (2n-1=31)
call      mid  left 

**Prova tu.** Cambia l'array: il sottoarray massimo può avere un solo elemento o attraversare il punto medio; le chiamate sono sempre $2n-1$.

In [19]:
%%writefile prova_maxsub_dc.cpp
#define NOMAIN
#include "maxsub_dc.cpp"

int main()
{
    // Cambia l'array: quanti elementi ha il sottoarray massimo? Le chiamate sono sempre 2n-1.
    int b[] = {4, -1, 2, 1, -5, 4, -7, 3, 3};
    fmsCalls = 0;
    Triple t = FindMaximumSubarray(b, 0, 8);
    printf("%s %d\n", Tr(t).c_str(), fmsCalls);
    return 0;
}

Writing prova_maxsub_dc.cpp


In [20]:
!g++ -std=c++17 -O2 -o prova_maxsub_dc prova_maxsub_dc.cpp && ./prova_maxsub_dc

(0, 3, 6) 17


## 6. Kadane

**Kadane**$(a,n)$ — *Input*: un array $a[1..n]$ di $n$ interi. *Output*: gli indici $bl\le br$ di un sottoarray di somma massima e la somma $best$. Una sola scansione: $cur$ è la somma massima di un sottoarray che **termina** nella posizione corrente e $cl$ il suo primo indice; se $cur<0$ si riparte da $a[i]$, altrimenti si estende. $\Theta(n)$ passi, memoria extra $\Theta(1)$. (Nel capitolo 18 si trova la versione compatta che restituisce solo la somma.)

Sull'esempio della compravendita di azioni (variazioni giornaliere $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$) si stampa la tabella del capitolo: una riga per iterazione, con i valori alla fine dell'iterazione (indici da 0, la riga $i=0$ è l'inizializzazione). Il risultato $(7,10,43)$, cioè $(8,11,43)$ con indici da 1, coincide con quello dell'algoritmo divide et impera.

In [21]:
%%writefile kadane.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

struct Triple
{
    int i, j, sum;
};

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Restituisce (bl, br, best): un sottoarray di somma massima di a[0..n-1]
Triple Kadane(int *a, int n)
{
    int best = a[0];
    int bl = 0;
    int br = 0;
    int cur = a[0];             // somma massima di un sottoarray che termina in i
    int cl = 0;
    int i = 1;
    while (i < n)
    {
        if (cur < 0)
        {
            cur = a[i];         // riparte
            cl = i;
        }
        else
        {
            cur = cur + a[i];   // estende
        }
        if (cur > best)
        {
            best = cur;
            bl = cl;
            br = i;
        }
        i = i + 1;
    }
    return {bl, br, best};
}

// Come sopra, ma stampa una riga per iterazione
Triple KadaneTrace(int *a, int n)
{
    int best = a[0], bl = 0, br = 0;
    int cur = a[0], cl = 0;
    printf("%3d %5d  %-24s %4d %3d %5d  (%d, %d)\n", 0, a[0], "inizializzazione", cur, cl, best, bl, br);
    for (int i = 1; i < n; i++)
    {
        const char *action;
        if (cur < 0)
        {
            cur = a[i];
            cl = i;
            action = "riparte (cur<0)";
        }
        else
        {
            cur = cur + a[i];
            action = "estende";
        }
        if (cur > best)
        {
            best = cur;
            bl = cl;
            br = i;
        }
        printf("%3d %5d  %-24s %4d %3d %5d  (%d, %d)\n", i, a[i], action, cur, cl, best, bl, br);
    }
    return {bl, br, best};
}

#ifndef NOMAIN
int main()
{
    int a[] = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
    int n = 16;
    printf("a = %s\n", Fmt(a, n).c_str());
    printf("%3s %5s  %-24s %4s %3s %5s  (bl, br)\n", "i", "a[i]", "azione", "cur", "cl", "best");
    Triple res = KadaneTrace(a, n);
    printf("risultato (%d, %d, %d)   (%d, %d, %d con indici da 1)\n", res.i, res.j, res.sum, res.i + 1, res.j + 1, res.sum);
    int b[] = {-2, 5, -1, 3};
    Triple t = Kadane(b, 4);
    printf("a = %s  Kadane(a, 4) = (%d, %d, %d)\n", Fmt(b, 4).c_str(), t.i, t.j, t.sum);
    return 0;
}
#endif

Writing kadane.cpp


In [22]:
!g++ -std=c++17 -O2 -o kadane kadane.cpp && ./kadane

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
  i  a[i]  azione                    cur  cl  best  (bl, br)
  0    13  inizializzazione           13   0    13  (0, 0)
  1    -3  estende                    10   0    13  (0, 0)
  2   -25  estende                   -15   0    13  (0, 0)
  3    20  riparte (cur<0)            20   3    20  (3, 3)
  4    -3  estende                    17   3    20  (3, 3)
  5   -16  estende                     1   3    20  (3, 3)
  6   -23  estende                   -22   3    20  (3, 3)
  7    18  riparte (cur<0)            18   7    20  (3, 3)
  8    20  estende                    38   7    38  (7, 8)
  9    -7  estende                    31   7    38  (7, 8)
 10    12  estende                    43   7    43  (7, 10)
 11    -5  estende                    38   7    43  (7, 10)
 12   -22  estende                    16   7    43  (7, 10)
 13    15  estende                    31   7    43  (7, 10)
 14    -4  estende                   

**Prova tu.** Cambia l'array (per esempio con soli numeri negativi): Kadane riparte a ogni passo e trova il massimo elemento.

In [23]:
%%writefile prova_kadane.cpp
#define NOMAIN
#include "kadane.cpp"

int main()
{
    // Prova con un array di soli numeri negativi: Kadane riparte a ogni passo e trova il massimo elemento.
    int c[] = {-3, -1, -2, -5};
    Triple t = Kadane(c, 4);
    printf("(%d, %d, %d)\n", t.i, t.j, t.sum);
    return 0;
}

Writing prova_kadane.cpp


In [24]:
!g++ -std=c++17 -O2 -o prova_kadane prova_kadane.cpp && ./prova_kadane

(1, 1, -1)
